2025_04 【必須スキル】マッピング表作成

In [39]:
import pandas as pd
import numpy as np
import os
import re

In [40]:
print(os.getcwd())

/app/src


In [41]:
# DBからエクスポートした必須スキルデータを取得
df = pd.read_csv("/app/data/cleansing_skill.csv")

In [42]:
# データの確認
df.head()

,id,company_name,required_skills,required_skills_tmp
0,106,トランスコスモス株式会社,必要な資格・条件\n特になし\n<推奨される要件>\n■ネイティブレベルの日本語スキル(日本...,必要な資格・条件\n特になし\n<推奨される要件>\n■ネイティブレベルの日本語スキル(日本...
1,5,株式会社クリーク・アンド・リバー社,求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...,求める人材: \n【応募要件（MUST）】\n▼下記のすべてのご経験・スキルをお持ちの方\n...
2,136,リクルーティング・パートナーズ株式会社 人材紹介事業部,NaN,NaN
3,115,株式会社クリーク・アンド・リバー社,求める人材: \n【求めるスキル・経験】※必須\n・SQLを使ったデータ抽出・集計の経験\n...,求める人材: \n【求めるスキル・経験】※必須\n・SQLを使ったデータ抽出・集計の経験\n...
4,53,ハロー・テクノ株式会社,求めている人材\n◆Webシステムの実務経験がある方\n◆AWSやPHP、C言語などを使用し...,求めている人材\n◆Webシステムの実務経験がある方\n◆AWSやPHP、C言語などを使用し...


In [43]:
# required_skills_tmpのデータを改行を基準に分割する
required_skills_tmp_split = df['required_skills_tmp'].dropna().str.split('\n').explode()

In [44]:
# 分割したデータの確認
required_skills_tmp_split.head(10)

0                              必要な資格・条件
0                                  特になし
0                             <推奨される要件>
0         ■ネイティブレベルの日本語スキル(日本語能力試験N1相当)
0    ■ビジネスレベルの英語スキル *面接は日本語・英語で実施いたします。
0           ■エクセルを使用したデータ作成・分析スキル\n<歓迎>
0                           ■BIツールの使用経験
0                 ■コンタクトセンター/サポートセンター経験
0        ■海外渡航の経験がある方(留学や旅行、ワーキングホリデー等)
0                    求めている人材の情報量は適切ですか？
Name: required_skills_tmp, dtype: object

In [45]:
# 分割後の行数の確認
print(len(required_skills_tmp_split))

2251


In [46]:
# 出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
十分                                    157
不足                                    157
求めている人材の情報量は適切ですか？                    157
求める人材:                                112
＜必須条件＞                                 26
                                     ... 
・Oracle EBS会計またはSCM領域の経験（両方あると尚可）       1
・シニアアナリストまたは導入コンサルタント経験 5年以上            1
・事業開発やPMI経験                             1
・人材紹介業・派遣業での経験                          1
・ロジカルにコミュニケーションしながら周りを巻き込む力\n✅歓迎条件      1
Name: count, Length: 1175, dtype: int64


In [47]:
# カンマなどで区切って、単語が複数含まれていそうなので更に分割する
required_skills_tmp_split = required_skills_tmp_split.str.split(r'[\t ,、・：]').explode()
required_skills_tmp_split = required_skills_tmp_split.str.split()

print(required_skills_tmp_split.head(10))

0                            [必要な資格]
0                               [条件]
0                             [特になし]
0                        [<推奨される要件>]
0    [■ネイティブレベルの日本語スキル(日本語能力試験N1相当)]
0                   [■ビジネスレベルの英語スキル]
0                          [*面接は日本語]
0                      [英語で実施いたします。]
0                  [■エクセルを使用したデータ作成]
0                      [分析スキル\n<歓迎>]
Name: required_skills_tmp, dtype: object


In [48]:
# 分割後の行数の確認
print(len(required_skills_tmp_split))

4885


In [49]:
# 出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
[]                      1039
[十分]                     157
[求めている人材の情報量は適切ですか？]     157
[不足]                     157
[求める人材:]                 112
                        ... 
[分析スキル\n<歓迎>]              1
[■エクセルを使用したデータ作成]          1
[英語で実施いたします。]              1
[*面接は日本語]                  1
[■ビジネスレベルの英語スキル]           1
Name: count, Length: 1963, dtype: int64


In [50]:
# 不要なワードのリストを作成
stopwords = [
    '必要な資格','条件','特になし','不足','十分','求める人材:','<推奨される要件>','求めている人材の情報量は適切ですか？','＜必須条件＞'
]

In [58]:
# リスト形式を文字列に変換
required_skills_tmp_split = required_skills_tmp_split.apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
)

# 前後の空白を削除
required_skills_tmp_split = required_skills_tmp_split.str.strip()

# stopwordに含まれているワードを削除
required_skills_tmp_split = required_skills_tmp_split[~required_skills_tmp_split.isin(stopwords)]

# 空白行とnanを削除
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != '']
required_skills_tmp_split =required_skills_tmp_split[required_skills_tmp_split != 'nan']

In [59]:
# stopwords の内容を確認
print("Stopwords:", stopwords)

# required_skills_tmp_split のサンプルデータを確認
print("Sample data from required_skills_tmp_split:", required_skills_tmp_split.head(10).tolist())

Stopwords: ['必要な資格', '条件', '特になし', '不足', '十分', '求める人材:', '<推奨される要件>', '求めている人材の情報量は適切ですか？', '＜必須条件＞']
Sample data from required_skills_tmp_split: ['■ネイティブレベルの日本語スキル(日本語能力試験N1相当)', '■ビジネスレベルの英語スキル', '*面接は日本語', '英語で実施いたします。', '■エクセルを使用したデータ作成', '分析スキル\\n<歓迎>', '■BIツールの使用経験', '■コンタクトセンター/サポートセンター経験', '■海外渡航の経験がある方(留学や旅行', 'ワーキングホリデー等)']


In [60]:
# ワードの出現頻度の確認
word_counts = required_skills_tmp_split.value_counts()
print(word_counts)

required_skills_tmp
求めている人材                  20
Python                   18
Ruby                     17
＋                        16
経験】                      16
                         ..
スタートアップ〜メガベンチャーでの就業経験     1
機械学習の知識                   1
統計学の知識                    1
Redash利用経験                1
機械学習の基礎理解                 1
Name: count, Length: 1949, dtype: int64
